In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Function Vectors Replication Notebook

This notebook replicates the key experiments from the "Function Vectors in Large Language Models" paper (Todd et al., ICLR 2024).

## Overview

The paper investigates whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning.

## Key Hypothesis:
A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.

In [2]:
# Core imports and setup
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from baukit import TraceDict
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Any
from sklearn.model_selection import train_test_split
from pathlib import Path

# Disable gradient computation for inference
torch.set_grad_enabled(False)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Set seed for reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)
print("Seed set to 42")

Using device: cuda
GPU: NVIDIA H200 NVL
Seed set to 42


In [3]:
# ============================================
# DATASET UTILITIES
# ============================================

class ICLDataset:
    """Dataset class for in-context learning experiments."""
    
    def __init__(self, data):
        if isinstance(data, str):
            self.data = pd.read_json(data)
        elif isinstance(data, dict):
            self.data = pd.DataFrame(data)
        else:
            self.data = data
        self.data = self.data[['input', 'output']]
    
    def __getitem__(self, idx):
        if isinstance(idx, int):
            return self.data.iloc[idx].to_dict()
        elif isinstance(idx, slice) or isinstance(idx, (list, np.ndarray)):
            return self.data.iloc[idx].to_dict(orient='list')
        elif isinstance(idx, str):
            return self.data[idx].tolist()
        raise ValueError(f"Invalid index type: {type(idx)}")
    
    def __len__(self):
        return len(self.data)


def split_dataset(dataset: ICLDataset, test_size: float = 0.3, seed: int = 42):
    """Split dataset into train, valid, and test sets."""
    train_data, valid_data = train_test_split(dataset.data, test_size=test_size, random_state=seed)
    test_data, valid_data = train_test_split(valid_data, test_size=test_size, random_state=seed)
    return {
        'train': ICLDataset(train_data.to_dict(orient='list')),
        'valid': ICLDataset(valid_data.to_dict(orient='list')),
        'test': ICLDataset(test_data.to_dict(orient='list'))
    }


def load_task_dataset(task_name: str, data_root: str = '/net/scratch2/smallyan/function_vectors_eval/dataset_files',
                      test_size: float = 0.3, seed: int = 32):
    """Load a task dataset."""
    for folder in ['abstractive', 'extractive']:
        file_path = os.path.join(data_root, folder, f'{task_name}.json')
        if os.path.exists(file_path):
            dataset = ICLDataset(file_path)
            return split_dataset(dataset, test_size=test_size, seed=seed)
    raise FileNotFoundError(f"Dataset {task_name} not found")


# ============================================
# PROMPT UTILITIES  
# ============================================

def build_prompt_data(word_pairs: Dict, query_pair: Dict = None, add_bos: bool = True,
                      shuffle_outputs: bool = False, prefixes: Dict = None, separators: Dict = None):
    """Build prompt data structure for ICL experiments."""
    if prefixes is None:
        prefixes = {"input": "Q:", "output": "A:", "instructions": ""}
    if separators is None:
        separators = {"input": "\n", "output": "\n\n", "instructions": ""}
    
    if add_bos:
        prefixes = {k: (v if k != 'instructions' else '<|endoftext|>' + v) for k, v in prefixes.items()}
    
    if query_pair is not None:
        query_pair = {k: (v[0] if isinstance(v, list) else v) for k, v in query_pair.items()}
    
    inputs = word_pairs.get('input', [])
    outputs = word_pairs.get('output', [])
    
    if shuffle_outputs and len(outputs) > 0:
        outputs = np.random.permutation(outputs).tolist()
    
    examples = [{'input': ' ' + str(inp), 'output': ' ' + str(out)} for inp, out in zip(inputs, outputs)]
    query_with_space = {k: ' ' + str(v) for k, v in query_pair.items()} if query_pair else None
    
    return {
        'instructions': '',
        'prefixes': prefixes,
        'separators': separators,
        'examples': examples,
        'query_target': query_with_space
    }


def construct_prompt(prompt_data: Dict, query: str = None) -> str:
    """Construct the full ICL prompt string."""
    if query is None and prompt_data['query_target'] is not None:
        query = prompt_data['query_target']['input']
    if isinstance(query, list):
        query = query[0]
    
    prompt = prompt_data['prefixes']['instructions'] + prompt_data['instructions'] + prompt_data['separators']['instructions']
    for ex in prompt_data['examples']:
        prompt += prompt_data['prefixes']['input'] + ex['input'] + prompt_data['separators']['input']
        prompt += prompt_data['prefixes']['output'] + ex['output'] + prompt_data['separators']['output']
    prompt += prompt_data['prefixes']['input'] + query + prompt_data['separators']['input']
    prompt += prompt_data['prefixes']['output']
    return prompt


# Test dataset loading
dataset = load_task_dataset('antonym', seed=0)
print(f"Dataset loaded - Train: {len(dataset['train'])}, Valid: {len(dataset['valid'])}, Test: {len(dataset['test'])}")
print(f"Sample: {dataset['train'][0]}")

Dataset loaded - Train: 1678, Valid: 216, Test: 504
Sample: {'input': 'limitless', 'output': 'limited'}


## Model Loading

Load GPT-J 6B - the smallest model used in the original experiments.

In [4]:
# Load GPT-J model and tokenizer
print("Loading GPT-J 6B model...")
model_name = 'EleutherAI/gpt-j-6b'

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, low_cpu_mem_usage=True, torch_dtype=torch.float16).to(device)

# Model configuration
model_config = {
    'n_heads': model.config.n_head,
    'n_layers': model.config.n_layer,
    'resid_dim': model.config.n_embd,
    'name_or_path': model.config.name_or_path,
    'attn_hook_names': [f'transformer.h.{layer}.attn.out_proj' for layer in range(model.config.n_layer)],
    'layer_hook_names': [f'transformer.h.{layer}' for layer in range(model.config.n_layer)],
    'prepend_bos': False
}

print(f"\nModel loaded successfully!")
print(f"  Layers: {model_config['n_layers']}")
print(f"  Heads: {model_config['n_heads']}")
print(f"  Hidden dim: {model_config['resid_dim']}")

# Edit layer for FV intervention (typically ~L/3)
EDIT_LAYER = 9
print(f"\nEdit layer for FV intervention: {EDIT_LAYER}")

Loading GPT-J 6B model...


`torch_dtype` is deprecated! Use `dtype` instead!


In [5]:
# Verify model is loaded
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")
print("Model ready for inference!")

In [6]:
# Wait for model - simple test
print("Testing...")

In [7]:
print("check")

In [8]:
x = 1

In [9]:
print("Kernel responsive test")

In [10]:
print("test2")

In [11]:
print("hello")

In [12]:
2+2

In [13]:
print("test")

In [14]:
print("model check")

pytorch_model.bin:   0%|          | 0.00/24.2G [00:00<?, ?B/s]

In [15]:
print("Checking download progress...")

In [16]:
print("status")

In [17]:
print("check")

In [18]:
print("model")

In [19]:
print("progress")

In [20]:
print("waiting")

In [21]:
print("check")

In [22]:
print("test")

In [23]:
print("still loading...")

In [24]:
print("model status?")

In [25]:
1+1

In [26]:
print("check")

In [27]:
print("check2")

In [28]:
print("status")

In [29]:
print("x")

In [30]:
print("waiting for model")

In [31]:
print("still waiting")

In [32]:
print("test")

In [33]:
print("check")

In [34]:
print("model check")

In [35]:
print("waiting...")

In [36]:
print("status check")

In [37]:
print("check")

In [38]:
print("checking model loading")

In [39]:
print("model")

In [40]:
print("checking...")

In [41]:
print("test123")

In [42]:
print("wait")

In [43]:
print("loading...")

In [44]:
print("checking")

In [45]:
print("model")

In [46]:
print("test")

In [47]:
print("continue checking")

In [48]:
print("model loading")

In [49]:
print("check status")

In [50]:
print("test")

In [51]:
print("hello")

In [52]:
print("check model")

In [53]:
print("model ready?")

In [54]:
print("status")

In [55]:
print("hello")

In [56]:
print("checking")

In [57]:
print("test")

In [58]:
print("checking model status")

In [59]:
print("waiting")

In [60]:
print("test")

In [61]:
print("check")

In [62]:
print("status")

In [63]:
print("model")

In [64]:
print("hello")

In [65]:
print("model ready?")

In [66]:
print("loading check")

In [67]:
print("model status?")

In [68]:
print("test")

In [69]:
print("check")

In [70]:
print("checking")

In [71]:
print("status")

In [72]:
print("model")

In [73]:
print("hello")

In [74]:
print("checking model status...")

In [75]:
print("is model ready?")

In [76]:
print("test")

In [77]:
print("model loading progress")

In [78]:
print("checking")

In [79]:
print("still downloading...")

In [80]:
print("model")

In [81]:
print("status")

In [82]:
print("check")

In [83]:
print("hello")

In [84]:
print("test")

In [85]:
print("check")

In [86]:
print("status")

In [87]:
print("model loading")

In [88]:
print("waiting for model")

In [89]:
print("check")

In [90]:
print("x")

In [91]:
print("y")

In [92]:
print("check")

In [93]:
print("test")

In [94]:
print("status")

In [95]:
print("model")

In [96]:
print("waiting")

In [97]:
print("hello")

In [98]:
print("checking")

In [99]:
print("model status")

In [100]:
print("model check")

In [101]:
print("test")

In [102]:
print("checking")

In [103]:
print("downloading...")

In [104]:
print("check")

In [105]:
print("model")

In [106]:
print("test")

In [107]:
print("status")

In [108]:
print("hello")

In [109]:
print("waiting")

In [110]:
print("check")

In [111]:
print("test")

In [112]:
print("checking")